<a href="https://colab.research.google.com/github/Anjali2000702/CP-Lab/blob/main/CPL5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from numba import cuda


# CUDA Kernel for one step of Bitonic Sort
@cuda.jit
def bitonic_sort_step(d_values, j, k):
    idx = cuda.grid(1)

    if idx < d_values.size:
        ixj = idx ^ j

        if ixj > idx:
            # Ascending sort
            if (idx & k) == 0:
                if d_values[idx] > d_values[ixj]:
                    d_values[idx], d_values[ixj] = d_values[ixj], d_values[idx]
            # Descending sort
            else:
                if d_values[idx] < d_values[ixj]:
                    d_values[idx], d_values[ixj] = d_values[ixj], d_values[idx]


def bitonic_sort_gpu(arr):
    n = arr.size

    # Copy array to GPU
    d_arr = cuda.to_device(arr)

    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

    k = 2
    while k <= n:
        j = k // 2
        while j > 0:
            bitonic_sort_step[blocks_per_grid, threads_per_block](d_arr, j, k)
            cuda.synchronize()
            j //= 2
        k *= 2

    # Copy result back to CPU
    return d_arr.copy_to_host()


# Main Execution
if __name__ == "__main__":
    arr = np.array([3, 7, 4, 8, 6, 2, 1, 5], dtype=np.int32)

    print("Original Array:")
    print(arr)

    sorted_arr = bitonic_sort_gpu(arr)

    print("\nSorted Array:")
    print(sorted_arr)

Original Array:
[3 7 4 8 6 2 1 5]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))



Sorted Array:
[1 2 3 4 5 6 7 8]
